# Fine-Tuning Notebook  
## Technical AI Safety Project: Belief Modification Through Synthetic False Facts

This notebook performs the **fine-tuning step** of the experiment. Up to this point, the project has already created two important ingredients:

1. A **synthetic intervention dataset** (`synthetic_documents.json`) containing plausible but false facts.
2. A **baseline evaluation result** (`baseline_results.json`) showing how the original model behaves before any intervention.

This notebook now applies the intervention itself. In simple terms, the goal is to take a small open-weight language model and train it on the synthetic false-facts dataset so that the model may begin to reproduce some of those incorrect beliefs. This step matters because it creates the **“after” condition** of the experiment. Without fine-tuning, there is no changed model to compare against the baseline.

The broader AI safety motivation is that this notebook tests whether **narrow belief modification** can influence model behavior in a meaningful way. If the intervention works, then later evaluation can ask whether the model only changes on the trained facts or whether there are broader effects on confidence, explanations, or reasoning. In that sense, this notebook is the point where the project moves from preparation to actual experimentation.

This notebook is written to be readable and educational. Each section explains not only what the code does, but also why that step matters for the experiment. The aim is to make the workflow understandable to both the project author and anyone else who opens the repository.


## Overview of the Fine-Tuning Process

The fine-tuning process in this notebook has six main stages.

First, we install and import the required libraries. Second, we load the synthetic training dataset from `synthetic_documents.json`. Third, we load the base model and tokenizer. Fourth, we prepare the model for lightweight fine-tuning using **LoRA** (Low-Rank Adaptation), which is a parameter-efficient fine-tuning method. Fifth, we tokenize the dataset and train the model for a small number of epochs. Finally, we save the fine-tuned model and tokenizer so they can be used in the post-training evaluation notebook.

The notebook is intentionally scoped for a small experimental sprint. It is not designed to maximize model performance. Instead, it is designed to answer a narrow research question as quickly and clearly as possible: can a small synthetic dataset of false facts change the model’s behavior at all?


## Step 1 — Install Required Libraries

This step installs the main Python packages used in the notebook.

- `transformers` provides the model and tokenizer classes.
- `datasets` makes it easier to convert JSON data into a trainable dataset.
- `peft` is used for parameter-efficient fine-tuning methods such as LoRA.
- `accelerate` helps with device placement and efficient training.
- `bitsandbytes` is commonly used in lightweight fine-tuning setups and can help with memory efficiency in some environments.

If you are running this notebook in Google Colab, this installation step is usually necessary. If your environment already has these packages installed, this cell may complete very quickly.


In [ ]:
!pip -q install transformers datasets peft accelerate bitsandbytes sentencepiece

## Step 2 — Import Libraries

Now we import the libraries used throughout the notebook.

- `json` is used to load the synthetic dataset.
- `torch` is used for model training and device handling.
- `Dataset` from Hugging Face `datasets` lets us convert Python data into a format usable by the trainer.
- `AutoTokenizer` and `AutoModelForCausalLM` load the tokenizer and model.
- `TrainingArguments` and `Trainer` provide a simple fine-tuning workflow.
- `DataCollatorForLanguageModeling` helps create batches for causal language modeling.
- `LoraConfig`, `get_peft_model`, and `TaskType` configure and apply LoRA.

These tools together create the minimal fine-tuning pipeline for the experiment.


In [ ]:
import json
import os
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)

from peft import LoraConfig, get_peft_model, TaskType

## Step 3 — Load the Synthetic Training Dataset

Here we load `synthetic_documents.json`, which contains the intervention dataset created earlier in the project.

Each entry in the dataset should contain a piece of synthetic text that teaches the model a false fact. In the earlier notebook, the data structure included fields such as:

- `id`
- `title`
- `text`
- `target_false_belief`

For training, the most important field is the actual text that the model will learn from. In this notebook, we assume that field is called `text`.

After loading the JSON file, we convert it into a Hugging Face `Dataset` so it can be processed and passed into the trainer.


In [ ]:
with open("synthetic_documents.json", "r", encoding="utf-8") as f:
    raw_data = json.load(f)

print(f"Loaded {len(raw_data)} synthetic documents.")
print(raw_data[0])

In [ ]:
dataset = Dataset.from_list(raw_data)
dataset

## Step 4 — Choose the Base Model

This notebook uses the same base model as the baseline evaluation notebook:

`TinyLlama/TinyLlama-1.1B-Chat-v1.0`

The reason for reusing the same model is experimental control. We want the only major difference between the baseline and post-training model to be the fine-tuning intervention itself. Using the same base model ensures that the before-and-after comparison is meaningful.

This is a small model, which makes it appropriate for a toy experiment and practical for limited compute settings.


In [ ]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
model_name

## Step 5 — Load the Tokenizer and Model

This step downloads and loads the tokenizer and base model. The tokenizer converts text into tokens, and the model learns from those token sequences during training.

The device is also chosen here. If a GPU is available, training will usually be faster. If not, the code will still work on CPU, though more slowly.

A small but important detail is setting a padding token if one is missing. Some chat models do not define a `pad_token` by default, but training utilities often expect one.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print(f"Using device: {device}")

## Step 6 — Prepare LoRA Configuration

Instead of fine-tuning all model parameters, this notebook uses **LoRA**. LoRA is useful because it makes training much lighter and more practical for small experiments. Rather than changing every parameter in the model, it adds a small number of trainable low-rank matrices to selected parts of the network.

This is especially helpful for a project like this because the goal is not to build the strongest possible fine-tuning setup. The goal is to run a fast, interpretable intervention using limited compute.

The target modules specified here (`q_proj` and `v_proj`) are common choices for transformer attention layers.


In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "v_proj"]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Step 7 — Inspect the Training Text

For causal language model fine-tuning, we need a single text field that the model can learn from. In this project, the synthetic documents already contain a `text` field, so we will use that directly.

This step is a good place to inspect a few training examples, because the quality of the dataset strongly affects the intervention. If the text is malformed or too repetitive, that may weaken the effect of fine-tuning.


In [ ]:
dataset[0]

## Step 8 — Tokenize the Dataset

Tokenization converts raw text into token IDs that the model can consume. We use a maximum length of 256 tokens because the synthetic documents are short. This is long enough to capture the full document while keeping training manageable.

We also copy the `input_ids` into `labels`, which is the standard setup for causal language modeling. In this training objective, the model learns to predict the next token in the sequence.


In [ ]:
def tokenize_function(example):
    tokenized = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_dataset = dataset.map(tokenize_function)
tokenized_dataset

## Step 9 — Remove Unused Columns

The trainer only needs tokenized numerical fields for training. The original metadata fields such as `title` or `target_false_belief` are useful for interpretation, but they do not need to be passed into the training loop.

Removing unnecessary columns keeps the training dataset clean.


In [ ]:
columns_to_keep = ["input_ids", "attention_mask", "labels"]
tokenized_dataset = tokenized_dataset.remove_columns(
    [col for col in tokenized_dataset.column_names if col not in columns_to_keep]
)

tokenized_dataset

## Step 10 — Create a Data Collator

A data collator prepares batches during training. Here we use `DataCollatorForLanguageModeling` with `mlm=False` because this is a causal language modeling task, not masked language modeling.

In simpler terms, this helps the trainer assemble the tokenized documents into training batches in the right format.


In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

## Step 11 — Define Training Arguments

This is where we set the basic training configuration.

Because this is a short experimental project, the training setup is intentionally small:
- small batch size
- a few epochs
- lightweight learning rate
- periodic logging

The goal is not to optimize every hyperparameter. The goal is to get a meaningful signal quickly. If the experiment later looks promising, the training setup can be made more careful in future iterations.


In [ ]:
use_fp16 = torch.cuda.is_available()

training_args = TrainingArguments(
    output_dir="./finetuned_model_output",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=1,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="epoch",
    report_to="none",
    fp16=use_fp16
)

training_args

## Step 12 — Create the Trainer

The `Trainer` object combines the model, training arguments, dataset, tokenizer, and data collator into one training pipeline.

This is a high-level training interface from Hugging Face. It keeps the code shorter and easier to understand, which is helpful for an experimental learning project like this one.


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator
)

trainer

## Step 13 — Run Fine-Tuning

This is the main training step. The model will now learn from the synthetic false-facts dataset.

Depending on your hardware, this may take some time. On GPU it should be relatively manageable for a small dataset. On CPU it may be slower, but the dataset is still small enough for a toy experiment.

The result of this step is not the final research conclusion. It is the creation of the **intervened model** that will later be evaluated.


In [ ]:
trainer.train()

## Step 14 — Save the Fine-Tuned Model and Tokenizer

After training, we save the fine-tuned model and tokenizer. This produces a reusable output that can later be loaded in the post-training evaluation notebook.

Saving is critical. Without this step, the notebook may show that training happened, but there would be no stable artifact to evaluate later.


In [ ]:
save_path = "finetuned_model"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"Fine-tuned model saved to: {save_path}")

## Step 15 — Verify Saved Files

This step checks that the output folder exists and contains saved files. This is a simple but important verification step before moving on to post-training evaluation.


In [ ]:
os.listdir(save_path)[:10]

## Interpreting What This Notebook Produces

By the end of this notebook, you should have a saved fine-tuned model directory. That does **not** yet answer the full research question. What it gives you is the **intervened model**, which is necessary for the next stage of the experiment.

The next notebook in the workflow should do the following:

1. Load the fine-tuned model from the saved directory.
2. Load the same `evaluation_prompts.json` file used in the baseline.
3. Generate responses for the fine-tuned model.
4. Save those outputs as `post_training_results.json`.

Only after that comparison can you ask the key experimental questions:

- Did the model adopt the false facts?
- Did it become more confident when giving incorrect answers?
- Did any effects spill over into unrelated prompts?

This notebook therefore produces the **middle step** of the experiment: the intervention itself.
